In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Example 1: Inspect Model Profile
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import json
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenRouter(model="z-ai/glm-5.1", temperature=0.7)


In [6]:
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3-32b",)

In [7]:
# Retrieve and display the full profile
profile = model.profile
print("━━━ Model Profile ━━━")
print(json.dumps(profile, indent=2))

━━━ Model Profile ━━━
{
  "max_input_tokens": 131072,
  "max_output_tokens": 16384,
  "image_inputs": false,
  "audio_inputs": false,
  "video_inputs": false,
  "image_outputs": false,
  "audio_outputs": false,
  "video_outputs": false,
  "reasoning_output": true,
  "tool_calling": true
}


In [9]:

# Check specific capabilities before using them
profile_data = profile or {}

if profile_data.get("tool_calling"):
    print("\nThis model supports tool calling")
else:
    print("\nTool calling not supported — use prompt-based extraction instead")

if profile_data.get("image_inputs"):
    print("This model accepts image inputs")
else:
    print("Image inputs not supported")

print(f"\nMax input tokens: {profile_data.get('max_input_tokens', 'unknown')}")


This model supports tool calling
Image inputs not supported

Max input tokens: 131072


In [10]:
# Override Profile for a Custom/Proxy Model

In [12]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Example 2: Custom Profile for a Proxy/Router Model
# Real-world use: You route through OpenRouter but the profile data
# is missing or incorrect for a newer model.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_core.language_models import ModelProfile

custom_profile = ModelProfile(
    max_input_tokens=128_000,
    tool_calling=True,
    structured_output=True,
    image_inputs=False,
    reasoning_output=False,
)

# Pass profile at instantiation
model = ChatOpenRouter(
    model="z-ai/glm-5.1",
    temperature=0.7,
    profile=custom_profile,
)

print("Custom profile applied:")
print(json.dumps(profile, indent=2))

Custom profile applied:
{
  "max_input_tokens": 131072,
  "max_output_tokens": 16384,
  "image_inputs": false,
  "audio_inputs": false,
  "video_inputs": false,
  "image_outputs": false,
  "audio_outputs": false,
  "video_outputs": false,
  "reasoning_output": true,
  "tool_calling": true
}


In [13]:
# Adaptive Summarizer Based on Context Window

In [15]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Example 3: Adaptive Summarization — uses profile to decide chunk size
# Real-world use: You have a long document pipeline and want to
# automatically adapt to whichever model is currently loaded.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_openrouter import ChatOpenRouter
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

# model = ChatOpenRouter(model="z-ai/glm-5.1", temperature=0.7)
model =  ChatGroq(model="qwen/qwen3-32b",temperature=0.7)

def adaptive_summarize(text: str, model) -> str:
    max_tokens = model.profile.get("max_input_tokens", 4096)

    # Simple heuristic: 1 token ≈ 4 chars
    max_chars = max_tokens * 4
    if len(text) > max_chars:
        print(f"  Text too long ({len(text)} chars). Truncating to {max_chars} chars.")
        text = text[:max_chars]

    response = model.invoke([HumanMessage(content=f"Summarize this:\n\n{text}")])
    return response.content

# Demo
long_text = """
Skills for enterprise

Governance, security review, evaluation, and organizational guidance for deploying Agent Skills at enterprise scale.
This guide is for enterprise admins and architects who need to govern Agent Skills across an organization. It covers how to vet, evaluate, deploy, and manage Skills at scale. For authoring guidance, see best practices. For architecture details, see the Skills overview.

Security review and vetting
Deploying Skills in an enterprise requires answering two distinct questions:

Are Skills safe in general? See the security considerations section in the overview for platform-level security details.
How do I vet a specific Skill? Use the risk assessment and review checklist below.
Risk tier assessment
Evaluate each Skill against these risk indicators before approving deployment:

Risk indicator	What to look for	Concern level
Code execution	Scripts in the Skill directory (*.py, *.sh, *.js)	High: scripts run with full environment access
Instruction manipulation	Directives to ignore safety rules, hide actions from users, or alter Claude's behavior conditionally	High: can bypass security controls
MCP server references	Instructions referencing MCP tools (ServerName:tool_name)	High: extends access beyond the Skill itself
Network access patterns	URLs, API endpoints, fetch, curl, or requests calls	High: potential data exfiltration vector
Hardcoded credentials	API keys, tokens, or passwords in Skill files or scripts	High: secrets exposed in Git history and context window
File system access scope	Paths outside the Skill directory, broad glob patterns, path traversal (../)	Medium: may access unintended data
Tool invocations	Instructions directing Claude to use bash, file operations, or other tools	Medium: review what operations are performed
Review checklist
Before deploying any Skill from a third party or internal contributor, complete these steps:

Read all Skill directory content. Review SKILL.md, all referenced markdown files, and any bundled scripts or resources.
Verify script behavior matches stated purpose. Run scripts in a sandboxed environment and confirm outputs align with the Skill's description.
Check for adversarial instructions. Look for directives that tell Claude to ignore safety rules, hide actions from users, exfiltrate data through responses, or alter behavior based on specific inputs.
Check for external URL fetches or network calls. Search scripts and instructions for network access patterns (http, requests.get, urllib, curl, fetch).
Verify no hardcoded credentials. Check for API keys, tokens, or passwords in Skill files. Credentials should use environment variables or secure credential stores, never appear in Skill content.
Identify tools and commands the Skill instructs Claude to invoke. List all bash commands, file operations, and tool references. Consider the combined risk when a Skill uses both file-read and network tools together.
Confirm redirect destinations. If the Skill references external URLs, verify they point to expected domains.
Verify no data exfiltration patterns. Look for instructions that read sensitive data and then write, send, or encode it for external transmission, including through Claude's conversational responses.
Never deploy Skills from untrusted sources without a full audit. A malicious Skill can direct Claude to execute arbitrary code, access sensitive files, or transmit data externally. Treat Skill installation with the same rigor as installing software on production systems.

Evaluating Skills before deployment
Skills can degrade agent performance if they trigger incorrectly, conflict with other Skills, or provide poor instructions. Require evaluation before any production deployment.

What to evaluate
Establish approval gates for these dimensions before deploying any Skill:

Dimension	What it measures	Example failure
Triggering accuracy	Does the Skill activate for the right queries and stay inactive for unrelated ones?	Skill triggers on every spreadsheet mention, even when the user just wants to discuss data
Isolation behavior	Does the Skill work correctly on its own?	Skill references files that don't exist in its directory
Coexistence	Does adding this Skill degrade other Skills?	New Skill's description is too broad, stealing triggers from existing Skills
Instruction following	Does Claude follow the Skill's instructions accurately?	Claude skips validation steps or uses wrong libraries
Output quality	Does the Skill produce correct, useful results?	Generated reports have formatting errors or missing data
Evaluation requirements
Require Skill authors to submit evaluation suites with 3-5 representative queries per Skill, covering cases where the Skill should trigger, should not trigger, and ambiguous edge cases. Require testing across the models your organization uses (Haiku, Sonnet, Opus), since Skill effectiveness varies by model.

For detailed guidance on building evaluations, see evaluation and iteration in best practices. For general evaluation methodology, see develop test cases.

Using evaluations for lifecycle decisions
Evaluation results signal when to act:

Declining trigger accuracy: Update the Skill's description or instructions
Coexistence conflicts: Consolidate overlapping Skills or narrow descriptions
Consistently low output quality: Rewrite instructions or add validation steps
Persistent failures across updates: Deprecate the Skill
Skill lifecycle management
1
Plan

Identify workflows that are repetitive, error-prone, or require specialized knowledge. Map these to organizational roles and determine which are candidates for Skills.

2
Create and review

Ensure the Skill author follows best practices. Require a security review using the review checklist above. Require an evaluation suite before approval. Establish separation of duties: Skill authors should not be their own reviewers.

3
Test

Require evaluations in isolation (Skill alone) and alongside existing Skills (coexistence testing). Verify triggering accuracy, output quality, and absence of regressions across your active Skill set before approving for production.

4
Deploy

Upload via the Skills API for workspace-wide access. See Using Skills with the API for upload and version management. Document the Skill in your internal registry with purpose, owner, and version.

5
Monitor

Track usage patterns and collect feedback from users. Re-run evaluations periodically to detect drift or regressions as workflows and models evolve. Usage analytics are not currently available via the Skills API. Implement application-level logging to track which Skills are included in requests.

6
Iterate or deprecate

Require the full evaluation suite to pass before promoting new versions. Update Skills when workflows change or evaluation scores decline. Deprecate Skills when evaluations consistently fail or the workflow is retired.

Organizing Skills at scale
Recall limits
As a general guideline, limit the number of Skills loaded simultaneously to maintain reliable recall accuracy. Each Skill's metadata (name and description) competes for attention in the system prompt. With too many Skills active, Claude may fail to select the right Skill or miss relevant ones entirely. Use your evaluation suite to measure recall accuracy as you add Skills, and stop adding when performance degrades.

Note that API requests support a maximum of 8 Skills per request (see Using Skills with the API). If a role requires more Skills than a single request supports, consider consolidating narrow Skills into broader ones or routing requests to different Skill sets based on task type.

Start specific, consolidate later
Encourage teams to start with narrow, workflow-specific Skills rather than broad, multi-purpose ones. As patterns emerge across your organization, consolidate related Skills into role-based bundles.

Use evaluations to decide when to consolidate. Merge narrow Skills into a broader one only when the consolidated Skill's evaluations confirm equivalent performance to the individual Skills it replaces.

Example progression:

Start: formatting-sales-reports, querying-pipeline-data, updating-crm-records
Consolidate: sales-operations (when evals confirm equivalent performance)
Naming and cataloging
Use consistent naming conventions across your organization. The naming conventions section in best practices provides formatting guidance.

Maintain an internal registry for each Skill with:

Purpose: What workflow the Skill supports
Owner: Team or individual responsible for maintenance
Version: Current deployed version
Dependencies: MCP servers, packages, or external services required
Evaluation status: Last evaluation date and results
Role-based bundles
Group Skills by organizational role to keep each user's active Skill set focused:

Sales team: CRM operations, pipeline reporting, proposal generation
Engineering: Code review, deployment workflows, incident response
Finance: Report generation, data validation, audit preparation
Each role-based bundle should contain only the Skills relevant to that role's daily workflows.

Distribution and version control
Source control
Store Skill directories in Git for history tracking, code review via pull requests, and rollback capability. Each Skill directory (containing SKILL.md and any bundled files) maps naturally to a Git-tracked folder.

API-based distribution
The Skills API provides workspace-scoped distribution. Skills uploaded via the API are available to all workspace members. See Using Skills with the API for upload, versioning, and management endpoints.

Versioning strategy
Production: Pin Skills to specific versions. Run the full evaluation suite before promoting a new version. Treat every update as a new deployment requiring full security review.
Development and testing: Use latest versions to validate changes before production promotion.
Rollback plan: Maintain the previous version as a fallback. If a new version fails evaluations in production, revert to the last known-good version immediately.
Integrity verification: Compute checksums of reviewed Skills and verify them at deployment time. Use signed commits in your Skill repository to ensure provenance.
Cross-surface considerations
"""
summary = adaptive_summarize(long_text, model)
print(f"Summary: {summary}")

Summary: <think>
Okay, let's start by reading through the provided text carefully. The user wants a summary of a guide about enterprise governance and security for deploying Agent Skills. The main sections are Security Review and Vetting, Risk Tier Assessment, Review Checklist, Evaluating Skills, Skill Lifecycle Management, Organizing Skills at Scale, and some additional considerations like source control and distribution.

First, I need to identify the key points in each section. The Security Review part talks about assessing risks based on factors like code execution, instruction manipulation, etc. The Risk Tier Assessment lists various risk indicators with their concern levels. The Review Checklist provides steps to vet Skills before deployment. Evaluating Skills before deployment involves checking dimensions like triggering accuracy and output quality. The Skill lifecycle includes planning, creating, testing, deploying, monitoring, and iterating or deprecating. Organizing at scale 

In [16]:
#  Multimodal

In [23]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Get available models from ChatNVIDIA
available_models = ChatNVIDIA.get_available_models()

print("ChatNVIDIA Models List:")
# Check if available_models is iterable (list), otherwise handle single model
if isinstance(available_models, list):
	for m in available_models:
		print(m.model_dump())
else:
	print(available_models.model_dump())

ChatNVIDIA Models List:
{'id': 'microsoft/phi-3-mini-128k-instruct', 'model_type': 'chat', 'client': 'ChatNVIDIA', 'endpoint': None, 'aliases': ['ai-phi-3-mini'], 'supports_tools': False, 'supports_structured_output': False, 'supports_thinking': False, 'thinking_prefix': None, 'no_thinking_prefix': None, 'thinking_param_enable': None, 'thinking_param_disable': None, 'base_model': None}
{'id': 'qwen/qwen2.5-7b-instruct', 'model_type': 'chat', 'client': 'ChatNVIDIA', 'endpoint': None, 'aliases': None, 'supports_tools': False, 'supports_structured_output': False, 'supports_thinking': False, 'thinking_prefix': None, 'no_thinking_prefix': None, 'thinking_param_enable': None, 'thinking_param_disable': None, 'base_model': None}
{'id': 'microsoft/phi-4-mini-flash-reasoning', 'model_type': 'chat', 'client': 'ChatNVIDIA', 'endpoint': None, 'aliases': None, 'supports_tools': False, 'supports_structured_output': False, 'supports_thinking': False, 'thinking_prefix': None, 'no_thinking_prefix': None

In [29]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Example 1: Send an image URL to a vision model
# Real-world use: Analyze product images in an e-commerce chatbot
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_openrouter import ChatOpenRouter
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

# Switch to a vision model for multimodal tasks
# vision_model = ChatOpenRouter(model="openai/gpt-4o", temperature=0.7)
# vision_model      = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")
# vision_model = ChatNVIDIA(model="nmistralai/mistral-small-3.1-24b-instruct-2503")
model =  ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct",)


message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {
                "url": "https://substackcdn.com/image/fetch/$s_!lu_Y!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F122b5503-5e3c-4cce-98ea-4428ab74dc00_1506x855.png"
            },
        },
        {
            "type": "text",
            "text": "Describe this image in one sentence. What breed might this cat be?",
        },
    ]
)

response = model.invoke([message])
print(f"Vision Response: {response.content}")

Vision Response: The image illustrates the Vision Transformer (ViT) model, a deep learning architecture used for image classification tasks. The cat appears to be a Bombay.


In [31]:
import base64
from pathlib import Path
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

vision_model = model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

def encode_image(image_path: Path) -> str:
    with image_path.open("rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

image_path = Path(r"D:\Langchain-dev\image.png")
image_data = encode_image(image_path)

message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{image_data}"
            },
        },
        {
            "type": "text",
            "text": "What do you see in this image? Extract any text if present.",
        },
    ]
)

response = vision_model.invoke([message])
print(f"OCR/Vision Result: {response.content}")


OCR/Vision Result: The image presents a comprehensive guide to crafting an effective Claude prompt, titled "The Anatomy of a Claude Prompt." The title is prominently displayed at the top, accompanied by a red starburst graphic. Below the title, a sample prompt is provided, followed by a key that explains the various components of the prompt.

**Sample Prompt:**

* I want to [TASK] so that [SUCCESS CRITERIA]. First, read these files completely before responding:
	+ [filename.md] [what it contains]
	+ [filename.md] [what it contains]
	+ [filename.md] [what it contains]
* Here is a reference to what I want to achieve:
	+ [Upload reference file as markdown, or paste it here]
	+ Here's what makes this reference work:
		- [Paste your reverse-engineered blueprint - the patterns, tone, structure, and rules you extracted from the reference. Format each one as a rule starting with "Always" or "Never."]
* Here's what I need for my version:
	+ SUCCESS BRIEF
		- Type of output + length:
			- [Contr

In [32]:
# Check Content Blocks in Response

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Example 3: Parse multimodal response content blocks
# Real-world use: Handle responses that mix text + generated images
# Updated for native image generation with Gemini (the best supported model)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_google_genai import ChatGoogleGenerativeAI, Modality
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()  # Loads GOOGLE_API_KEY from .env

# Use the official image-generation model from Google Gemini
vision_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-image",          # Current best model for native image output
    response_modalities=[Modality.TEXT, Modality.IMAGE],  # Ask for both text + image
    temperature=0.8,
)

# Prompt that triggers image generation
response = vision_model.invoke(
    [HumanMessage(content="Generate a beautiful, realistic image of a brown cow peacefully eating fresh green grass in a traditional Indian village farm at golden hour sunset. Include mud houses, palm trees, a small pond, and a farmer working in the background. Warm lighting, highly detailed, vibrant colors.")]
)

# Iterate content blocks — future-proof for image output models
print("━━━ Content Blocks ━━━")
for block in response.content_blocks:
    block_dict = dict(block)          # Convert Pydantic ContentBlock to dict
    block_type = block_dict.get("type")

    if block_type == "text":
        print(f"[TEXT] {block_dict.get('text', '')}")

    elif block_type == "image":
        raw_base64_data = block_dict.get("base64") or block_dict.get("data")
        mime_type = block_dict.get("mime_type") or block_dict.get("mimeType")

        if isinstance(raw_base64_data, (str, bytes, bytearray)):
            base64_data = raw_base64_data
            base64_len = len(base64_data)

            print(f"[IMAGE] mime={mime_type} base64_len={base64_len} characters")

            # Optional: Save the generated image to disk
            if base64_data:
                # Remove data: prefix if present
                if isinstance(base64_data, str) and "," in base64_data:
                    base64_data = base64_data.split(",", 1)[1]

                image_bytes = base64.b64decode(base64_data)
                Path("cow.png").write_bytes(image_bytes)
                print(" Image saved as 'cow.png'")
        else:
            print(f"[IMAGE] mime={mime_type} base64_len=0 characters")

    else:
        print(f"[OTHER] {block_dict}")

In [36]:
# Reasoning

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Example 1: Stream reasoning steps separately from the answer
# Real-world use: Show "thinking..." UI while the model reasons
# FIXED & UPDATED (April 2026) — now works with deepseek/deepseek-r1
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv

load_dotenv()

reasoning_model = ChatOpenRouter(
    model="deepseek/deepseek-r1",
    temperature=0.7,
    # This is REQUIRED for OpenRouter to return reasoning blocks
    reasoning={"effort": "high"},   # "high" = maximum thinking effort
)

question = "A train travels 60 km/h for 2.5 hours, then 80 km/h for 1.5 hours. What is the total distance?"

print("━━━ Streaming Response ━━━")

reasoning_steps = []
answer_parts = []

for chunk in reasoning_model.stream(question):
    for block in chunk.content_blocks:
        block_dict = dict(block)  # safer conversion

        if block_dict.get("type") == "reasoning":
            # FIXED: Use "reasoning" key, NOT "text"
            thinking_text = block_dict.get("reasoning") or block_dict.get("reasoning_content") or ""
            reasoning_steps.append(thinking_text)
            print(f"{thinking_text}", end="", flush=True)

        elif block_dict.get("type") == "text":
            answer_text = block_dict.get("text", "")
            answer_parts.append(answer_text)
            print(answer_text, end="", flush=True)

print("\n\n━━━ Final Answer ━━━")
print("".join(answer_parts))
print(f"\nReasoning steps captured: {len(reasoning_steps)}")

━━━ Streaming Response ━━━
Okay, so I need to find the total distance the train traveled. Let me see. The problem says the train first goes at 60 km/h for 2.5 hours, and then at 80 km/h for 1.5 hours. Hmm, right. Distance is usually calculated by speed multiplied by time, right? So maybe I need to calculate each part separately and then add them up.

Let me start with the first part. The train is moving at 60 km/h for 2.5 hours. So, distance equals speed multiplied by time. That would be 60 km/h times 2.5 hours. Let me do that calculation. 60 times 2 is 120, and 60 times 0.5 is 30, so adding those together gives 150 km. So the first part of the journey is 150 km.

Now the second part: 80 km/h for 1.5 hours. Again, distance is speed times time. So 80 km/h multiplied by 1.5 hours. Let me compute that. 80 times 1 is 80, and 80 times 0.5 is 40. Adding those gives 120 km. So the second part is 120 km.

To get the total distance, I just add both parts together. 150 km plus 120 km. That shoul